# YorkerAPI: a season of cricket in five minutes

This notebook loads the free YorkerAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: YorkerAPI (yorkerapi.com), under the [YorkerAPI Sample Data Licence](https://github.com/sports-data-hq/yorkerapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/yorkerapi-mcp/releases/download/season-pack/yorkerapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('ipl', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## Run scorers, minimum 150 balls faced

In [ ]:
b = t('ipl', 'batting')
b[b.balls_faced >= 150].sort_values('runs', ascending=False)[['player', 'innings', 'runs', 'average', 'strike_rate']].head(10)

## Fastest scorers: strike rate, minimum 150 balls

In [ ]:
b = t('ipl', 'batting')
b[b.balls_faced >= 150].sort_values('strike_rate', ascending=False)[['player', 'runs', 'balls_faced', 'strike_rate', 'sixes']].head(10)

## Teams by our Elo

In [ ]:
t('ipl', 'elo').sort_values('elo_end_of_season', ascending=False)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the YorkerAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=yorkerapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/cricket/get_ratings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the cricket skills from https://github.com/sports-data-hq/yorkerapi-mcp.